<a href="https://colab.research.google.com/github/mibucko/ml-product-categories/blob/main/product_categories_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is part of a machine learning project in which we are developing a model to predict the product category based on the input data. The documents are available on GitHub at the following link:
https://github.com/mibucko/ml-product-categories

1. Raw data gathering

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/mibucko/ml-product-categories/main/products.csv"
df = pd.read_csv(url)

1.1. Column names standardization

In [ ]:
df.columns = (df.columns.str.strip().str.lower().str.replace(r"[\s_]+", "_", regex=True).str.strip("_"))
print(df.columns)

Index(['product_id', 'product_title', 'merchant_id', 'category_label',
       'product_code', 'number_of_views', 'merchant_rating', 'listing_date'],
      dtype='object')


1.2. Brief view on data

In [ ]:
print("Number of rows:", len(df))
print("Sample rows:")
print(df.sample(5))

Number of rows: 35311
Sample rows:
       product_id                                      product_title  \
3231         3241              blackview p2 lite dual sim smartphone   
26570       37612                          lec t5039w fridge freezer   
31327       42980        zanussi zrt23103wa fridge freezers in white   
14903       24552  sharp r843slm 25 litre 900w double grill combi...   
18472       28583        aeg fse21200p integrated compact dishwasher   

       merchant_id   category_label product_code  number_of_views  \
3231            53    Mobile Phones   FT-9669-RF            396.0   
26570          129  Fridge Freezers   XW-3778-UL           4376.0   
31327          294  Fridge Freezers   WC-6145-AK             69.0   
14903          289       Microwaves   GF-5621-CV           2873.0   
18472          186      Dishwashers   YX-3635-UR           4947.0   

       merchant_rating listing_date  
3231               3.4    9/15/2022  
26570              1.1    7/19/2023  
313

1.3. We erase columns that definitely cannot influence product category: product_id, merchant_id, number_of_views, merchant_rating and listing_date.

In [ ]:
df = df.drop(columns=["product_id", "merchant_id", "number_of_views",
    "merchant_rating", "listing_date"])

2. Exploratory data analysis – EDA

2.1. What categories are present in dataset?

In [ ]:
print("Number of categories:", df["category_label"].nunique())
print(df["category_label"].value_counts())

Number of categories: 13
category_label
Fridge Freezers     5495
Washing Machines    4036
Mobile Phones       4020
CPUs                3771
TVs                 3564
Fridges             3457
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
fridge               123
CPU                   84
Mobile Phone          55
Name: count, dtype: int64


2.2. Is the product code related to the product category?

We choose two categories as an example. By visual inspection, we can see that there is no apparent correlation between the product code and the product category. Therefore, we also drop this column from the dataset.

In [ ]:
mobile_phones = df[df["category_label"] == "Mobile Phones"]
cameras = df[df["category_label"] == "Digital Cameras"]
print('mobile phones')
print(mobile_phones[["product_code", "category_label"]].sample(20))
print('cameras')
print(cameras[["product_code", "category_label"]].sample(20))

mobile phones
     product_code category_label
1717   MM-4844-BY  Mobile Phones
679    CY-6896-SD  Mobile Phones
4009   FV-1030-HP  Mobile Phones
3725   KU-3835-RT  Mobile Phones
749    VE-0381-EH  Mobile Phones
2235   KY-5727-WH  Mobile Phones
1154   XK-8770-JF  Mobile Phones
3181   XS-9712-GF  Mobile Phones
947    SL-7317-SF  Mobile Phones
3876   MJ-3431-QP  Mobile Phones
1111   HD-8992-VB  Mobile Phones
111    IP-6428-FK  Mobile Phones
936    CM-6057-QN  Mobile Phones
3978   LJ-8139-UB  Mobile Phones
1649   OY-7674-IV  Mobile Phones
2342   TC-7433-UO  Mobile Phones
2762   EC-8703-TN  Mobile Phones
2885   XN-0327-VL  Mobile Phones
295    WQ-6192-EE  Mobile Phones
2934   QG-0370-IW  Mobile Phones
cameras
      product_code   category_label
12940   QL-8182-XP  Digital Cameras
12449   RS-9369-CO  Digital Cameras
14061   HY-9191-DJ  Digital Cameras
13043   ZC-7966-TR  Digital Cameras
12488   FD-3779-WO  Digital Cameras
12288   BU-9687-CX  Digital Cameras
14131   XE-3056-FM  Digital Camer

In [ ]:
df = df.drop(columns=["product_code"])
print('Remaining columns:')
print(df.columns)

Remaining columns:
Index(['product_title', 'category_label'], dtype='object')


3. Data processing

3.1 Merging similar categories

In [ ]:
df["category_label"] = df["category_label"].replace({
    "Mobile Phone": "Mobile Phones",
    "CPU": "CPUs",
    "fridge": "Fridges"
})
print(df["category_label"].value_counts())

category_label
Fridge Freezers     5495
Mobile Phones       4075
Washing Machines    4036
CPUs                3855
Fridges             3580
TVs                 3564
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
Name: count, dtype: int64


3.2. Handling Missing Values - isna

There are 171 rows without a product title, so we drop these rows. There are also 43 rows without a product category. We remove these rows from the original dataset and store them separately in a dataset called "only_title", so that we can potentially use them later to test our model.

In [ ]:
print("product_title:", df["product_title"].isna().sum())
print("category_label:", df["category_label"].isna().sum())
only_title = df[df["category_label"].isna()]

product_title: 172
category_label: 44


3.2. Handling Missing Values - dropna

We drop rows with missing values from the original dataset.

In [ ]:
df = df.dropna(subset=["product_title", "category_label"])

4. Feature engineering

4.1 Brief look at product titles

In [ ]:
sample = df.sample(10).sort_values("category_label")
print(sample[["category_label", "product_title"]].to_string(index=False))

 category_label                                                   product_title
Digital Cameras                          sony hx350 cybershot schwarz dschx350b
Digital Cameras                          kodak fz51 digital camera fz51 bk mw01
    Dishwashers  smeg df6fabrd 60cm 50s style freestanding retro dishwasher red
    Dishwashers                  siemens iq500 sr456s00pe integrierbar 9stellen
       Freezers bosch gsn58aw30g freezer upright frost free 360 litres a logixx
       Freezers               237litre upright freezer frost free class a white
       Freezers                                bush bcf198l chest freezer white
     Microwaves                       zanussi zbm 17542 xa microwave/17 l/800 w
  Mobile Phones                                    nokia 7373 pink mobile phone
  Mobile Phones                                                      alcatel u3


4.2. Creating new Columns

We create five new features from the product title:

- word_count – number of words in the product title
- char_count – number of characters in the product title
- special_count – number of special characters in the product title
- words_with_digits – number of words containing digits in the title
- max_word_length – length of the longest word in the title

In [ ]:
df["word_count"] = df["product_title"].str.split().str.len()
df["char_count"] = df["product_title"].str.replace(" ", "").str.len()
df["special_count"] = df["product_title"].str.count(r"[^A-Za-z0-9\s]")
df["words_with_digits"] = df["product_title"].str.split().apply(
    lambda words: sum(any(char.isdigit() for char in word) for word in words))
df["max_word_length"] = df["product_title"].str.split().apply(
    lambda words: max(len(word) for word in words))

5. Data Preparation for Algorithm Training

5.1 Firstly we define seven experimental data sets:

In [ ]:
y = df["category_label"]

feature_sets = {
    "baseline": ["product_title"],
    "word_count": ["product_title", "word_count"],
    "char_count": ["product_title", "char_count"],
    "special_count": ["product_title", "special_count"],
    "words_with_digits": ["product_title", "words_with_digits"],
    "max_word_length": ["product_title", "max_word_length"],
    "all_features": [
        "product_title",
        "word_count",
        "char_count",
        "special_count",
        "words_with_digits",
        "max_word_length"
    ]
}

5.2 Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

5.3. TF-IDF Vectorization: We choose the unigram + bigram option

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True
)

X_train_text = tfidf.fit_transform(
    df.loc[train_idx, "product_title"]
)

X_test_text = tfidf.transform(
    df.loc[test_idx, "product_title"]
)

5.4 Scaling Numerical Features

In [ ]:
from sklearn.preprocessing import MinMaxScaler

numeric_features = [
    "word_count",
    "char_count",
    "special_count",
    "words_with_digits",
    "max_word_length"
]

scaler = MinMaxScaler()

X_train_num = scaler.fit_transform(
    df.loc[train_idx, numeric_features]
)

X_test_num = scaler.transform(
    df.loc[test_idx, numeric_features]
)

6. Training and Testing of the Algorithms

We have seven experimental datasets. We will perform training and testing using five different algorithms. Each Colab cell is dedicated to one algorithm and includes all seven experimental datasets.

6.1 Logistic Regression

In [78]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from scipy.sparse import hstack
from sklearn.metrics import confusion_matrix

feature_sets = {
    "Baseline": [],
    "Word count": ["word_count"],
    "Char count": ["char_count"],
    "Special count": ["special_count"],
    "Words with digits": ["words_with_digits"],
    "Max word length": ["max_word_length"],
    "All features": numeric_features
}

for name, features in feature_sets.items():

    if len(features) == 0:
        X_train_final = X_train_text
        X_test_final = X_test_text
    else:
        indices = [numeric_features.index(f) for f in features]

        X_train_final = hstack([
            X_train_text,
            X_train_num[:, indices]
        ])

        X_test_final = hstack([
            X_test_text,
            X_test_num[:, indices]
        ])

    model = LogisticRegression(max_iter=1000)

    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)

    print(f"\n{'='*60}")
    print(f"Logistic Regression - {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

    print("Confusion Matrix:")
    print(pd.DataFrame(
    cm,
    index=model.classes_,
    columns=model.classes_))


Logistic Regression - Baseline
                  precision    recall  f1-score   support

            CPUs       1.00      0.99      1.00       766
 Digital Cameras       1.00      0.99      0.99       538
     Dishwashers       0.93      0.96      0.95       681
        Freezers       0.99      0.93      0.96       440
 Fridge Freezers       0.96      0.94      0.95      1094
         Fridges       0.89      0.92      0.90       712
      Microwaves       0.99      0.95      0.97       466
   Mobile Phones       0.97      0.99      0.98       812
             TVs       0.97      0.99      0.98       708
Washing Machines       0.95      0.94      0.94       803

        accuracy                           0.96      7020
       macro avg       0.96      0.96      0.96      7020
    weighted avg       0.96      0.96      0.96      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               761                0            0         0   
Dig